In [ ]:
# Code to map Maven Lead elements to Origami Lead elements
# Using semantic similarity library
# Author: Sandeep Chintabathina
# March 2026

In [ ]:
# For each Maven Lead data element, check if there is an exact or close matching data element in the origami Lead package
# The output will list the maven element alongside the origami element and other supporting attributes

In [230]:
from sentence_transformers import SentenceTransformer


In [231]:
import pandas as pd
import numpy as np

In [232]:
# Read maven elements
df_maven = pd.read_csv('output/maven_all_with_mmg_and_cste.csv',na_filter=False)
len(df_maven)

9501

In [233]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9501 entries, 0 to 9500
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Model                  9501 non-null   object
 1   question_number        9501 non-null   int64 
 2   depth                  9501 non-null   int64 
 3   Question               9501 non-null   object
 4   Question.Package.ID    9501 non-null   object
 5   Question.Package.Name  9501 non-null   object
 6   Message                9501 non-null   object
 7   mmg_element            9501 non-null   object
 8   cste_element           9501 non-null   object
dtypes: int64(2), object(7)
memory usage: 668.2+ KB


In [234]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If not space, digits, uppercase letters, lowercase letters, or underscore, ignore it
        if not (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=''
        else:
            new_name+=ch
    return new_name.strip()

In [ ]:
# Clean up questions or data elements
df_maven.loc[:,'Question'] =[ clean_up(q) for q in df_maven.loc[:,'Question']]

In [236]:
df_maven.loc[:,'Model'].value_counts(dropna=False)

Model
General                  3866
STD                      2425
Coinfection               923
Hepatitis                 901
Childhood Lead            606
Cluster                   425
Lead investigation        255
STD EHR and screening      85
Aggregate                  15
Name: count, dtype: int64

In [237]:
df_maven_lead = df_maven[df_maven['Model']=='Childhood Lead']
len(df_maven_lead)

606

In [238]:
# No duplicates
df_maven_lead[df_maven_lead.duplicated(['Question'],keep=False)]

,Model,question_number,depth,Question,Question.Package.ID,Question.Package.Name,Message,mmg_element,cste_element


In [239]:
# Read origami elements
df_origami = pd.read_csv('origami/Custom__CaseLead.csv',na_filter=False)
len(df_origami)

783

In [240]:
df_origami.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 783 entries, 0 to 782
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Field                 783 non-null    object
 1   Type                  783 non-null    object
 2   Label                 783 non-null    object
 3   Required in Database  783 non-null    bool  
 4   Can't Protect         783 non-null    bool  
 5   FK Table              783 non-null    object
 6   CodeTypeID            783 non-null    object
 7   Static Codes          783 non-null    object
 8   Field Notes           783 non-null    object
 9   Global Comments       783 non-null    object
 10  Local Comments        783 non-null    object
dtypes: bool(2), object(9)
memory usage: 56.7+ KB


In [241]:
df_origami.loc[:,'Label'].value_counts(dropna=False)

Label
                                       382
HI_lead_18                               2
HI_lead_26                               2
Have they all been tested for lead?      2
HI_lead_8                                2
                                      ... 
HG_EXP_BROKEN_ITEM                       1
HG_EXP_EMISSIONS                         1
HG_EXP_PRODUCT                           1
HG_EXP_JOB_ACTIVITY                      1
HI_lead_13_details                       1
Name: count, Length: 398, dtype: int64

In [242]:
# Get the two columns of interest and non-blank values
df_origami = df_origami[['Field','Label']]
df_origami = df_origami[df_origami['Label']!='']
len(df_origami)

401

In [243]:
df_origami = df_origami.reset_index(drop=True)

In [244]:
# Original label to be used later for output
original_labels = list(df_origami.Label)
original_labels

['Case Lead ID',
 'Client',
 'Entry User',
 'Entry Date',
 'Modified User',
 'Modified Date',
 'IncidentID',
 '[lead_preg_blt]If pregnant, was blood lead testing',
 '[lead_under16]Is this a Child Case (<16 yrs) or Ad',
 '[lead_bl_specsource]Specimen Source',
 '[lead_above_3.5]Test result at or above 3.5 µg/dL',
 '[lead_bl_reas_testing]Reason for testing',
 '[lead_performing_lab_type]Performing lab type',
 '[lead_bl_test_funding]Test source of funding',
 '[lead_medicaid]Medicaid',
 '[lead_chelated]Chelation therapy administered',
 '[lead_chelate_type]Chelated type',
 '[lead_chel_fund]Source of funding for the chelatio',
 '[lead_dev_assessment]Developmental assessment cond',
 '[lead_household_lead]We they tested for lead',
 '[21522001]Abdominal Pain or Tenderness',
 '[86276007]Bleeding gums',
 '[14760008]Constipation',
 '[84229001]Fatigue',
 '[25064002]Headache',
 '[422400008]Vomiting',
 '[721294001]Hearing loss',
 '[163028000]High blood pressure',
 '[79890006]Loss of appetite',
 '[91175

In [245]:
# Process the column values (eliminate the values in square brackets)
df_origami.loc[:,'Label'] = [ label.split(']')[1].strip() if ']' in label else label.strip() for label in df_origami.loc[:,'Label']]

In [246]:
# Clean up column names in both data sets
df_origami.loc[:,'Label'] =[ clean_up(q) for q in df_origami.loc[:,'Label']]

In [247]:
df_origami.loc[:,'Label'].value_counts()

Label
HI_lead_26                            2
HI_lead_8                             2
HI_lead_18                            2
Investigation Completed               2
Have they all been tested for lead    2
                                     ..
HG_FILLING_REPAIR                     1
HG_FILLING                            1
HG_WORK_SITE                          1
HG_COSMETIC                           1
HI_lead_13_details                    1
Name: count, Length: 394, dtype: int64

In [248]:
# duplicates labels exist but they are labels to different questions, so keep them
df_origami[df_origami.duplicated(['Label'],keep=False)]

,Field,Label
67,CustomText6,Address
70,CustomText9,Name
73,CustomText12,Name
77,CustomText16,Address
100,CustomDate14,Investigation Completed
102,CustomDate16,Investigation Completed
231,CustomText93,HI_lead_8
320,CustomCode203ID,HI_lead_18
325,CustomCode208ID,HI_lead_26
326,CustomCode209ID,Have they all been tested for lead


In [249]:
# Do a semantic match between Maven Lead elements and Origami Lead labels

In [250]:
maven_elements = [str.replace(maven_de,'_',' ') for maven_de in df_maven_lead.Question]
origami_elements = [str.replace(col,'_',' ') for col in df_origami.Label]

In [251]:
origami_elements

['Case Lead ID',
 'Client',
 'Entry User',
 'Entry Date',
 'Modified User',
 'Modified Date',
 'IncidentID',
 'If pregnant was blood lead testing',
 'Is this a Child Case 16 yrs or Ad',
 'Specimen Source',
 'Test result at or above 35 gdL',
 'Reason for testing',
 'Performing lab type',
 'Test source of funding',
 'Medicaid',
 'Chelation therapy administered',
 'Chelated type',
 'Source of funding for the chelatio',
 'Developmental assessment cond',
 'We they tested for lead',
 'Abdominal Pain or Tenderness',
 'Bleeding gums',
 'Constipation',
 'Fatigue',
 'Headache',
 'Vomiting',
 'Hearing loss',
 'High blood pressure',
 'Loss of appetite',
 'Seizures',
 'Weight loss',
 'Other',
 'Do any teenages or adult',
 'Blood tested',
 'Are clothes cha',
 'Do they shower at work',
 'Do they wash hands',
 'Do they wash wo',
 'Does anyone in the home have a hobby i',
 'Does the child eat or chew on n',
 'Does the child eat dirt',
 'County',
 'Address type',
 'Type of Residence',
 'Ownership',
 'Cu

In [252]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [253]:
# Compute embeddings
embeddings_maven = model.encode(maven_elements)
embeddings_origami = model.encode(origami_elements)

In [254]:
print(len(embeddings_origami))
print(len(embeddings_maven))
embeddings_origami
embeddings_maven

401
606


array([[-0.0020506 , -0.04598626, -0.04524118, ...,  0.01019603,
         0.0600717 , -0.01581826],
       [-0.080986  ,  0.06571371, -0.00726671, ..., -0.03971622,
         0.06275451, -0.00660688],
       [-0.01527976,  0.01455189, -0.0803012 , ...,  0.00756959,
         0.01350452, -0.00275409],
       ...,
       [-0.01195142,  0.04254857,  0.08214842, ..., -0.01518289,
        -0.02453376,  0.0252025 ],
       [-0.06644832,  0.03604519,  0.10580117, ..., -0.01695742,
         0.02714118,  0.0212451 ],
       [-0.0375696 ,  0.05908274,  0.00948133, ...,  0.02978237,
        -0.01489797, -0.0133038 ]], dtype=float32)

In [255]:
# Compute cosine similarities
similarities = model.similarity(embeddings_maven,embeddings_origami)

In [256]:
similarities

tensor([[ 0.0258,  0.1144, -0.0019,  ...,  0.2399,  0.0640,  0.2023],
        [ 0.1138,  0.1608, -0.0025,  ...,  0.0504,  0.1517,  0.0842],
        [ 0.0338, -0.0344, -0.0397,  ...,  0.1266,  0.0640,  0.1591],
        ...,
        [ 0.1462,  0.2037,  0.1194,  ...,  0.1201,  0.1136,  0.1390],
        [ 0.1729,  0.1440,  0.1463,  ...,  0.0985,  0.1070,  0.1319],
        [ 0.1609,  0.0577,  0.0730,  ...,  0.0437,  0.0396,  0.0893]])

In [257]:
type(similarities)
len(similarities)

606

In [258]:
df_similarities = pd.DataFrame(similarities,index=df_maven_lead.Question,columns=original_labels)

In [259]:
df_similarities

,Case Lead ID,Client,Entry User,Entry Date,Modified User,Modified Date,IncidentID,"[lead_preg_blt]If pregnant, was blood lead testing",[lead_under16]Is this a Child Case (<16 yrs) or Ad,[lead_bl_specsource]Specimen Source,...,PROBABLE_LEAD_RESOURCES,HI_lead_6_no_children,HI_lead_17,HI_lead_19,HI_lead_20,HI_lead_24,HI_lead_28,HI_lead_8,lead_investigation_notes,HI_lead_13_details
Question,,,,,,,,,,,,,,,,,,,,,
ANY_OVER_5_OUTPUT,0.025803,0.114357,-0.001872,-0.041852,0.006444,-0.036348,0.040639,0.053973,-0.016145,0.078986,...,0.138921,0.259962,0.226058,0.198540,0.272845,0.235286,0.235718,0.239852,0.063962,0.202338
VENOUS_CONFIRMED_DISPLAY,0.113808,0.160789,-0.002502,0.032497,0.035942,0.068626,0.045402,0.300372,0.136487,0.192706,...,0.078033,0.069234,0.056359,0.076221,0.065648,0.078259,0.051564,0.050370,0.151701,0.084232
MAX_VENOUS_BLL_RANGE_OUTPUT,0.033788,-0.034410,-0.039742,-0.000766,-0.029528,-0.007219,-0.030989,0.224953,-0.015748,0.132655,...,0.096739,0.114929,0.130708,0.136197,0.146979,0.124870,0.118783,0.126571,0.064016,0.159142
MAX_VENOUS_BLL_OUTPUT,0.056942,-0.009140,-0.033509,-0.007262,-0.018913,0.007159,0.008254,0.205379,0.001018,0.135731,...,0.058997,0.097167,0.117021,0.112082,0.125881,0.101424,0.099759,0.104436,0.050632,0.141410
MAX_VENOUS_BLL_DATE_OUTPUT,0.059105,-0.038171,-0.045077,0.384399,0.001480,0.368358,0.009315,0.221654,0.101369,0.172937,...,0.030890,0.054688,0.118256,0.121773,0.101268,0.108117,0.116337,0.081822,0.065683,0.120927
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DR_OVERDUE_TEST_DATE,0.192236,-0.008107,-0.106171,0.315164,0.037759,0.367981,0.086776,0.333878,0.183493,0.156000,...,0.111712,0.131025,0.160527,0.178756,0.150856,0.164453,0.151854,0.121792,0.207612,0.146040
DR_MAILING_ADDRESS,0.216611,0.207069,0.260992,0.113655,0.286297,0.131592,0.246804,0.050145,0.172110,0.107888,...,0.048195,0.091866,0.151359,0.176754,0.160173,0.149623,0.175883,0.165202,0.106563,0.174280
DR_MAILING_CITY,0.146156,0.203670,0.119402,0.099299,0.180808,0.123937,0.229790,0.091015,0.204169,0.125451,...,0.073077,0.120737,0.150558,0.145556,0.121450,0.110293,0.137216,0.120115,0.113576,0.139016


In [260]:
df_similarities.to_csv('output/similarity_scores_lead.csv')

In [267]:
df_similarities.loc['DEVELOPMENTAL_SCREENING',:]

Case Lead ID                0.073111
Client                      0.064448
Entry User                  0.051299
Entry Date                  0.064655
Modified User               0.008360
                              ...   
HI_lead_24                  0.084902
HI_lead_28                  0.078631
HI_lead_8                   0.087034
lead_investigation_notes    0.129956
HI_lead_13_details          0.068488
Name: DEVELOPMENTAL_SCREENING, Length: 401, dtype: float32

In [268]:
df_maven_lead = df_maven_lead.reset_index(drop=True)
df_maven_lead.columns

Index(['Model', 'question_number', 'depth', 'Question', 'Question.Package.ID',
       'Question.Package.Name', 'Message', 'mmg_element', 'cste_element'],
      dtype='object')

In [269]:
df_output = df_maven_lead[['Model','depth','Question','Message','mmg_element','cste_element']]

In [ ]:
#For every maven element (index), identify origami elements (column) that is the best match

filtered=[]
for idx in df_similarities.index:
    # Looking for columns with score higher than 0.92
    v = df_similarities.loc[idx,:]>0.92

    # Get the column with the maximum score
    max=0.0
    max_col=''
    for i,col in enumerate(df_similarities.columns):
        if v.iloc[i]==True:
            if df_similarities.loc[idx,col] > max:
                max = df_similarities.loc[idx,col]
                max_col=col

    #filtered.append({col:df_similarities.loc[idx,col]})
    print(idx+','+max_col+':'+str(max)+'\n')
    filtered.append({'Question':idx,'target_label':max_col})



ANY_OVER_5_OUTPUT,:0.0

VENOUS_CONFIRMED_DISPLAY,:0.0

MAX_VENOUS_BLL_RANGE_OUTPUT,:0.0

MAX_VENOUS_BLL_OUTPUT,:0.0

MAX_VENOUS_BLL_DATE_OUTPUT,:0.0

BLL_DECLINE_OUTPUT,:0.0

BLL_TOTAL_DECLINING_OUTPUT,:0.0

LATE_FOLLOW_UP_OVER_RIDE,:0.0

TESTING_DUE_DATE_OUTPUT,:0.0

LATEST_VENOUS_BLL_OUTPUT,:0.0

LATEST_VENOUS_BLL_DATE_OUTPUT,:0.0

LATEST_VENOUS_BLL_UNDER_5_OUTPUT,:0.0

LATEST_VENOUS_BLL_UNDER_5_DATE_OUTPUT,:0.0

LATEST_BLL_OUTPUT,:0.0

LATEST_BLL_DATE_OUTPUT,:0.0

LATEST_BLL_TYPE_OUTPUT,:0.0

LATEST_VENOUS_BLL_UNDER_5_TRUE_FALSE_DISPLAY,:0.0

CASE_CLOSURE_REASON,:0.0

CASE_CLOSURE_DATE,:0.0

REPORTED_BLOOD_LEVEL,:0.0

EPI_REQUIRED,:0.0

PHN_SECTION_LOOKUP,:0.0

PHN_ASSIGNED,:0.0

CONTACT_POINT_POSTAL_CODE_CHECK_DISPLAY,:0.0

PRIMARY_HOME_ZIP_CHECK,:0.0

REPORTING_SOURCE,:0.0

CDC_REPORTED_DATE,:0.0

CDC_REPORTED_DATE_OPERATION,:0.0

LEAD_BROCHURE_DATE,:0.0

DATE_OF_CONTACT,:0.0

DATE_OF_CONTACT_DATE_TYPE,:0.0

DATE_OF_CONTACT_DATE_TYPE_OTHER_SPECIFY,:0.0

DATE_OF_CONTACT_PERSON_CONT

In [276]:
df_filtered = pd.DataFrame(filtered)
print(len(df_filtered))
df_filtered.loc[:,'target_label'].value_counts(dropna=False)

606


target_label
                                                      432
DEVELOPMENTAL_SCREENING_EIGHT_TWELVE_COMMUNICATION      8
DEVELOPMENTAL_SCREENING_SIXTEEN_TWENTY_GROSS_MOTOR      7
DEVELOPMENTAL_SCREENING_SIXTEEN_TWENTY_FINE_MOTOR       4
DEVELOPMENTAL_SCREENING_FIFTYFOUR_SIXTY_FINE_MOTOR      3
                                                     ... 
INTERIM_RELOCATION_ADDRESS_STREET                       1
INTERIM_RELOCATION_DATE                                 1
INTERIM_RELOCATION                                      1
INTERIM_MEASURES_IMPLEMENTED_PHONE                      1
CHECKLIST_INVESTIGATION_DATE                            1
Name: count, Length: 145, dtype: int64

In [277]:
# Merge df_filtered with df_origami, first set df_origami to original labels
df_origami.Label = original_labels


In [278]:
df_merge = pd.merge(df_filtered,df_origami,left_on='target_label',right_on='Label',how='left')

In [279]:
df_merge

,Question,target_label,Field,Label
0,ANY_OVER_5_OUTPUT,,NaN,NaN
1,VENOUS_CONFIRMED_DISPLAY,,NaN,NaN
2,MAX_VENOUS_BLL_RANGE_OUTPUT,,NaN,NaN
3,MAX_VENOUS_BLL_OUTPUT,,NaN,NaN
4,MAX_VENOUS_BLL_DATE_OUTPUT,,NaN,NaN
...,...,...,...,...
601,DR_OVERDUE_TEST_DATE,,NaN,NaN
602,DR_MAILING_ADDRESS,,NaN,NaN
603,DR_MAILING_CITY,,NaN,NaN
604,DR_MAILING_STATE,,NaN,NaN


In [280]:
# merge the above merged df with output dataframe
df_final = pd.merge(df_output,df_merge,on='Question')
df_final

,Model,depth,Question,Message,mmg_element,cste_element,target_label,Field,Label
0,Childhood Lead,0,ANY_OVER_5_OUTPUT,,,,,NaN,NaN
1,Childhood Lead,0,VENOUS_CONFIRMED_DISPLAY,,,,,NaN,NaN
2,Childhood Lead,0,MAX_VENOUS_BLL_RANGE_OUTPUT,,,,,NaN,NaN
3,Childhood Lead,0,MAX_VENOUS_BLL_OUTPUT,,,,,NaN,NaN
4,Childhood Lead,0,MAX_VENOUS_BLL_DATE_OUTPUT,,,,,NaN,NaN
...,...,...,...,...,...,...,...,...,...
601,Childhood Lead,0,DR_OVERDUE_TEST_DATE,,,,,NaN,NaN
602,Childhood Lead,0,DR_MAILING_ADDRESS,,,,,NaN,NaN
603,Childhood Lead,0,DR_MAILING_CITY,,,,,NaN,NaN
604,Childhood Lead,0,DR_MAILING_STATE,,,,,NaN,NaN


In [281]:
# Add, rename, remove columns as needed
df_final['target_table'] = 'Custom__CaseLead'
df_final=df_final.rename(columns={'Field':'target_field'})
df_final=df_final.drop('Label',axis=1)


In [282]:
df_final = df_final[["Model",'Question','target_table','target_field','target_label','mmg_element','cste_element','Message','depth']]
df_final

,Model,Question,target_table,target_field,target_label,mmg_element,cste_element,Message,depth
0,Childhood Lead,ANY_OVER_5_OUTPUT,Custom__CaseLead,NaN,,,,,0
1,Childhood Lead,VENOUS_CONFIRMED_DISPLAY,Custom__CaseLead,NaN,,,,,0
2,Childhood Lead,MAX_VENOUS_BLL_RANGE_OUTPUT,Custom__CaseLead,NaN,,,,,0
3,Childhood Lead,MAX_VENOUS_BLL_OUTPUT,Custom__CaseLead,NaN,,,,,0
4,Childhood Lead,MAX_VENOUS_BLL_DATE_OUTPUT,Custom__CaseLead,NaN,,,,,0
...,...,...,...,...,...,...,...,...,...
601,Childhood Lead,DR_OVERDUE_TEST_DATE,Custom__CaseLead,NaN,,,,,0
602,Childhood Lead,DR_MAILING_ADDRESS,Custom__CaseLead,NaN,,,,,0
603,Childhood Lead,DR_MAILING_CITY,Custom__CaseLead,NaN,,,,,0
604,Childhood Lead,DR_MAILING_STATE,Custom__CaseLead,NaN,,,,,0


In [283]:
df_final[df_final['target_field'].isna()]

,Model,Question,target_table,target_field,target_label,mmg_element,cste_element,Message,depth
0,Childhood Lead,ANY_OVER_5_OUTPUT,Custom__CaseLead,NaN,,,,,0
1,Childhood Lead,VENOUS_CONFIRMED_DISPLAY,Custom__CaseLead,NaN,,,,,0
2,Childhood Lead,MAX_VENOUS_BLL_RANGE_OUTPUT,Custom__CaseLead,NaN,,,,,0
3,Childhood Lead,MAX_VENOUS_BLL_OUTPUT,Custom__CaseLead,NaN,,,,,0
4,Childhood Lead,MAX_VENOUS_BLL_DATE_OUTPUT,Custom__CaseLead,NaN,,,,,0
...,...,...,...,...,...,...,...,...,...
601,Childhood Lead,DR_OVERDUE_TEST_DATE,Custom__CaseLead,NaN,,,,,0
602,Childhood Lead,DR_MAILING_ADDRESS,Custom__CaseLead,NaN,,,,,0
603,Childhood Lead,DR_MAILING_CITY,Custom__CaseLead,NaN,,,,,0
604,Childhood Lead,DR_MAILING_STATE,Custom__CaseLead,NaN,,,,,0


In [284]:
df_final.to_csv('output/maven_lead_origami_similarity_match.csv',index=False)